# AgriGuard v2 — training manual di Google Colab
Jalankan sembilan tahap dari atas ke bawah. Notebook memanggil CLI Task 1: **prepare → train → evaluate → export**. Pilih **Runtime → Change runtime type → T4 GPU**.

Pada pemakaian pertama, alur berhenti di review CSV sampai label dan kelompok kejadian/video selesai diperiksa. Setelah runtime reset, jalankan kembali dari tahap 1 dengan `EXPERIMENT_ID` dan `RUN_NAME` yang sama; data, konfigurasi, dan checkpoint dipulihkan dari Drive.

Label v2 bersifat netral: `tanpa_kerusakan_terlihat`, `kerusakan_ringan_terlihat`, `kerusakan_berat_terlihat`. Dataset penyakit hanya pendukung train. Hasil smoke test bukan metrik akurasi v2. Rincian sumber, anotasi, dan batas klaim: [panduan training](https://github.com/AbdulHalim26/agriguard/blob/main/ai-service/training/README.md).

## 1. Setup — cek GPU, clone repo, install dependensi
Script dipin ke commit Task 1 (`cd3b0c4`) agar flag dan kontrak konsisten saat resume. Instalasi memakai `training/requirements.txt`; setiap tahap dijalankan sebagai proses CLI baru.

In [ ]:
import csv
import getpass
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

REPO_URL = 'https://github.com/AbdulHalim26/agriguard.git'
REPO_REF = 'cd3b0c4'
REPO_DIR = Path('/content/agriguard')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU belum aktif. Pilih Runtime → Change runtime type → T4 GPU, lalu ulangi sel ini.')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError('Folder /content/agriguard sudah ada tetapi bukan clone Git; pilih REPO_DIR lain.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
TRAINING_DIR = REPO_DIR / 'ai-service' / 'training'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(TRAINING_DIR / 'requirements.txt')], check=True)
sys.path.insert(0, str(TRAINING_DIR))
from common import LABELS, file_sha256, load_checkpoint, load_manifest, read_json, utc_now, write_json
from prepare_dataset import extract_zip, public_url
import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch belum mendeteksi CUDA. Pastikan runtime Colab memakai GPU.')
DEVICE = 'cuda'

def run_cli(script, *arguments):
    command = [sys.executable, str(TRAINING_DIR / script), *map(str, arguments)]
    print(shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

print('Repo:', REPO_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. Mount Drive — dataset dan checkpoint persisten
Gunakan ID eksperimen yang sama sesudah reset. Untuk mengganti sumber/label/split, gunakan `EXPERIMENT_ID` baru; untuk mencoba parameter training baru, gunakan `RUN_NAME` baru. Checkpoint setiap epoch, dataset beku, metrik, dan ekspor disimpan di Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXPERIMENT_ID = 'experiment-001' # @param {type:'string'}
RUN_NAME = 'run-v2' # @param {type:'string'}
EPOCHS = 15 # @param {type:'integer'}
BATCH_SIZE = 32 # @param {type:'integer'}
WORKERS = 2 # @param {type:'integer'}
SEED = 42 # @param {type:'integer'}
if not all(re.fullmatch(r'[A-Za-z0-9_-]+', value) for value in (EXPERIMENT_ID, RUN_NAME)):
    raise ValueError('ID eksperimen/run hanya boleh berisi huruf, angka, underscore, atau tanda hubung.')
WORK = Path('/content/drive/MyDrive/agriguard-v2') / EXPERIMENT_ID
WORK.mkdir(parents=True, exist_ok=True)
SOURCES_PATH = WORK / 'sources.json'
CACHE_DIR = WORK / 'raw-cache'
INVENTORY_DIR = WORK / 'inventory'
ANNOTATIONS_DIR = WORK / 'annotations'
DATASET_DIR = WORK / 'dataset-v2'
RUN_DIR = WORK / RUN_NAME
SMOKE_MARKER = WORK / 'smoke-passed.json'
METRICS_PATH = RUN_DIR / 'metrics.json'
ONNX_PATH = RUN_DIR / 'model_v2.onnx'
METADATA_PATH = RUN_DIR / 'model_v2.metadata.json'
os.environ['TORCH_HOME'] = str(WORK / 'torch-cache')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
NOTEBOOK_STATE = WORK / 'notebook-state.json'
if NOTEBOOK_STATE.exists() and read_json(NOTEBOOK_STATE)['repo_commit'] != REPO_COMMIT:
    raise RuntimeError('Commit pipeline eksperimen ini berbeda; pakai commit semula atau ID eksperimen baru.')
write_json(NOTEBOOK_STATE, {'repo_commit': REPO_COMMIT})
print('Direktori persisten:', WORK, '\nCheckpoint:', RUN_DIR)


## 3. Dataset — Roboflow atau upload ZIP sendiri
Mode `roboflow`: simpan **`ROBOFLOW_API_KEY`** di Colab Secrets (ikon kunci), atau isi prompt tersembunyi. Workspace/project/versi di bawah adalah kandidat Wet Rice dari README, **bukan tiga kelas severity siap pakai**. Periksa kesesuaian foto dan versi; pilih export tanpa augmentasi offline. Jika export tidak tersedia, gunakan mode `upload_zip`.

Mode `upload_zip`: pilih satu ZIP gambar melalui dialog, isi URL asal dan lisensi sebenarnya. Gunakan `EXPERIMENT_ID` baru jika berpindah mode/sumber. Struktur folder boleh bertingkat; CSV inventory memakai path relatif terhadap root hasil ekstraksi. Gambar penyakit ditandai `disease_support` dan hanya masuk train; dataset utama tetap harus mencakup tiga label padi genangan/banjir. Sumber tambahan dapat ditambahkan di `sources.json` Drive sebelum prepare.

API key dan URL unduhan bertanda tangan hanya berada di memori/environment, bukan konfigurasi atau output notebook. Saat resume, konfigurasi dan gambar yang sudah ada di Drive digunakan kembali.

In [ ]:
DATA_SOURCE = 'roboflow' # @param ['roboflow', 'upload_zip']
ROBOFLOW_WORKSPACE = 'no-85bd3' # @param {type:'string'}
ROBOFLOW_PROJECT = 'wet-rice-lvtac' # @param {type:'string'}
ROBOFLOW_VERSION = 1 # @param {type:'integer'}
ROBOFLOW_FORMAT = 'folder' # @param {type:'string'}
SOURCE_LICENSE = 'CC-BY-4.0' # @param {type:'string'}
SOURCE_KIND = 'flood_paddy' # @param ['flood_paddy', 'disease_support']
OWN_SOURCE_URL = '' # @param {type:'string'}
OWN_SOURCE_NAME = 'Foto padi genangan milik tim' # @param {type:'string'}

if not SOURCES_PATH.exists():
    source = {'id': 'primary', 'enabled': True, 'license': SOURCE_LICENSE, 'kind': SOURCE_KIND,
              'annotations': 'annotations/primary.csv', 'labels_reviewed': False, 'reviewed_by': '', 'review_protocol': ''}
    if DATA_SOURCE == 'roboflow':
        source.update(name=f'Roboflow {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}',
                      url=f'https://universe.roboflow.com/{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}',
                      version=ROBOFLOW_VERSION, download_url_env='ROBOFLOW_DATASET_ZIP_URL')
    elif DATA_SOURCE == 'upload_zip':
        public_url(OWN_SOURCE_URL)
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1 or Path(next(iter(uploaded))).suffix.lower() != '.zip':
            raise ValueError('Pilih tepat satu file ZIP gambar.')
        payload = next(iter(uploaded.values()))
        upload_id = hashlib.sha256(payload).hexdigest()[:16]
        archive = WORK / 'uploads' / f'{upload_id}.zip'
        archive.parent.mkdir(parents=True, exist_ok=True)
        archive.write_bytes(payload)
        raw = WORK / 'raw' / upload_id
        if not raw.exists():
            raw.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.TemporaryDirectory(prefix='.extract-', dir=raw.parent) as temporary:
                extracted = Path(temporary) / 'images'
                extract_zip(archive, extracted)
                extracted.replace(raw)
        source.update(name=OWN_SOURCE_NAME, url=OWN_SOURCE_URL, version=upload_id, local_dir=str(raw))
        del uploaded, payload
    else:
        raise ValueError('DATA_SOURCE tidak dikenal.')
    public_url(source['url'])
    write_json(SOURCES_PATH, {'sources': [source]})

sources = read_json(SOURCES_PATH)
active_sources = [item for item in sources['sources'] if item.get('enabled', True)]
if not active_sources:
    raise ValueError('Aktifkan setidaknya satu sumber di sources.json.')
for source in active_sources:
    root = (WORK / source.get('local_dir', '/nonexistent-agriguard-source')).resolve()
    cached_root = CACHE_DIR / source['id'] / 'extracted'
    if root.is_dir():
        continue
    if cached_root.is_dir() and (CACHE_DIR / source['id'] / 'receipt.json').is_file():
        source['local_dir'] = str(cached_root)
        continue
    if source.get('download_url_env') != 'ROBOFLOW_DATASET_ZIP_URL':
        raise RuntimeError('Folder sumber belum ada. Lengkapi local_dir/konfigurasi download sumber tambahan di Drive.')
    # ROBOFLOW_API_KEY adalah placeholder secret; jangan menuliskan nilainya di sel.
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
    if not ROBOFLOW_API_KEY:
        try:
            from google.colab import userdata
            ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
        except Exception:
            ROBOFLOW_API_KEY = getpass.getpass('ROBOFLOW_API_KEY: ')
    if not ROBOFLOW_API_KEY:
        raise ValueError('ROBOFLOW_API_KEY belum diisi.')
    project_path = urllib.parse.urlsplit(source['url']).path.strip('/')
    endpoint = f'https://api.roboflow.com/{project_path}/{int(source["version"])}/{ROBOFLOW_FORMAT}'
    request_url = endpoint + '?' + urllib.parse.urlencode({'api_key': ROBOFLOW_API_KEY})
    try:
        with urllib.request.urlopen(request_url, timeout=120) as response:
            export_info = json.load(response)
    except (urllib.error.URLError, ValueError):
        raise RuntimeError('Export Roboflow gagal. Periksa key, project, versi/format, atau gunakan upload ZIP manual.') from None
    link = export_info.get('export', {}).get('link')
    if not link or urllib.parse.urlsplit(link).scheme != 'https':
        raise RuntimeError('Link ZIP Roboflow belum tersedia. Ulangi sel ini setelah export siap atau gunakan ZIP manual.')
    os.environ[source['download_url_env']] = link
    del ROBOFLOW_API_KEY, request_url, export_info, link
write_json(SOURCES_PATH, sources)
if not INVENTORY_DIR.exists():
    run_cli('prepare_dataset.py', '--sources', SOURCES_PATH, '--download', '--inventory-only',
            '--cache', CACHE_DIR, '--output', INVENTORY_DIR)
for source in active_sources:
    if not source.get('local_dir'):
        source['local_dir'] = str(CACHE_DIR / source['id'] / 'extracted')
    print(source['id'], '|', source['url'], '| inventory:', INVENTORY_DIR / f'{source["id"]}.csv')
write_json(SOURCES_PATH, sources)


## 4. Review inventory CSV → prepare dataset
Pertama jalankan dengan `REVIEW_COMPLETE=False` untuk melihat contoh dan mengunduh inventory. Isi kolom `label` dengan salah satu label v2, lalu **review `group_id`**: semua frame/crop dari satu kejadian/lahan/video harus satu kelompok, termasuk lintas sumber. Hapus baris gambar yang tidak relevan/tidak jelas. Jangan mengubah `path`; pertahankan `original_label`.

Simpan nama file sebagai `<source_id>.csv` (default `primary.csv`). Setelah review, isi penilai/protokol, set `REVIEW_COMPLETE=True`, jalankan ulang sel, dan upload CSV hasil review; bisa juga menaruhnya di folder `annotations/` pada Drive. Prepare memvalidasi label, provenance, deduplikasi, dan split 70/15/15. Setiap label data utama memerlukan kelompok independen untuk train/val/test. Pada resume, dataset yang sudah beku diverifikasi dan digunakan kembali.

In [ ]:
REVIEW_COMPLETE = False # @param {type:'boolean'}
REVIEWER = '' # @param {type:'string'}
REVIEW_PROTOCOL = '' # @param {type:'string'}
sources = read_json(SOURCES_PATH)
active_sources = [item for item in sources['sources'] if item.get('enabled', True)]
if not (DATASET_DIR / 'manifest.json').is_file():
    if not REVIEW_COMPLETE:
        from google.colab import files
        print('Label yang valid:', LABELS)
        for source in active_sources:
            inventory_csv = INVENTORY_DIR / f'{source["id"]}.csv'
            with inventory_csv.open(newline='', encoding='utf-8-sig') as handle:
                for index, row in enumerate(csv.DictReader(handle)):
                    if index >= 5:
                        break
                    print(row)
            files.download(str(inventory_csv))
        raise RuntimeError('Review CSV terlebih dahulu, kemudian isi penilai/protokol dan set REVIEW_COMPLETE=True.')
    if not REVIEWER.strip() or not REVIEW_PROTOCOL.strip():
        raise ValueError('Isi REVIEWER dan REVIEW_PROTOCOL berdasarkan review sebenarnya.')
    ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    missing = [item for item in active_sources if not (WORK / item['annotations']).is_file()]
    if missing:
        from google.colab import files
        print('Upload CSV:', [f'{item["id"]}.csv' for item in missing])
        uploaded = files.upload()
        for source in missing:
            name = f'{source["id"]}.csv'
            if name not in uploaded:
                raise ValueError(f'CSV hasil review belum diupload: {name}')
            destination = WORK / source['annotations']
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(uploaded[name])
        del uploaded
    for source in active_sources:
        source.update(labels_reviewed=True, reviewed_by=REVIEWER.strip(), review_protocol=REVIEW_PROTOCOL.strip())
    write_json(SOURCES_PATH, sources)
    run_cli('prepare_dataset.py', '--sources', SOURCES_PATH, '--cache', CACHE_DIR,
            '--output', DATASET_DIR, '--seed', SEED)
MANIFEST = load_manifest(DATASET_DIR)
if MANIFEST['purpose'] != 'model_training':
    raise RuntimeError('Dataset ini adalah fixture smoke. Gunakan dataset hasil review untuk training penuh.')
configured = {item['id']: item for item in active_sources}
if set(configured) != {item['id'] for item in MANIFEST['dataset_sources']}:
    raise RuntimeError('Sumber berubah sesudah split dibekukan; gunakan eksperimen baru.')
for original in MANIFEST['dataset_sources']:
    current = configured[original['id']]
    if any(current.get(key) != original.get(key) for key in ('url', 'license', 'kind', 'version', 'reviewed_by', 'review_protocol')):
        raise RuntimeError('Atribusi/review sumber berubah; gunakan eksperimen baru.')
    if file_sha256(WORK / current['annotations']) != original['annotation_sha256']:
        raise RuntimeError('Anotasi berubah sesudah prepare; gunakan eksperimen baru.')
print('Dataset beku:', MANIFEST['dataset_size'], '| split:', MANIFEST['split']['id'])
print(json.dumps(MANIFEST['distribution'], indent=2, ensure_ascii=False))


## 5. Smoke test — subset batch kecil dahulu
Uji **3 epoch, maksimum 2 batch train/validation per epoch**, batch maksimum 8 gambar dari dataset hasil review. Flag `--smoke-test --max-batches 2` membatasi pekerjaan dan menandai checkpoint sebagai smoke, dengan folder terpisah. Ini memeriksa pemanasan head dan unfreeze backbone sebelum training penuh. Hasilnya bukan angka akurasi untuk pitch.

Helper berikut hanya menyusun pemanggilan `train.py`, memilih checkpoint epoch terakhir, dan mengenali run yang sudah selesai. Setelah reset, konfigurasi harus sama dengan run semula. Smoke yang sudah lulus digunakan kembali bila manifest dan commit masih sama.

In [ ]:
def run_training(output, epochs, batch_size, workers, *, smoke=False):
    flags = ['--dataset', DATASET_DIR, '--output', output, '--epochs', epochs,
             '--batch-size', batch_size, '--workers', workers, '--seed', SEED,
             '--device', DEVICE, '--pretrained', 'imagenet']
    if smoke:
        flags += ['--smoke-test', '--max-batches', 2]
    checkpoints = sorted(output.glob('epoch_*.pt'), key=lambda path: int(path.stem.split('_')[-1]))
    completed = False
    if checkpoints:
        last = load_checkpoint(checkpoints[-1])
        config = last['training_config']
        expected = {'epochs': epochs, 'batch_size': batch_size, 'workers': workers, 'seed': SEED,
                    'pretrained': 'imagenet', 'max_batches': 2 if smoke else None, 'amp': DEVICE == 'cuda'}
        if any(config.get(key) != value for key, value in expected.items()):
            raise RuntimeError('Parameter resume berbeda; gunakan parameter run semula atau RUN_NAME baru.')
        if last['dataset']['manifest_sha256'] != MANIFEST['manifest_sha256']:
            raise RuntimeError('Checkpoint berasal dari dataset/split berbeda.')
        completed = last['epoch'] >= epochs or last['stale_epochs'] >= config['patience']
        if not completed:
            flags += ['--resume', checkpoints[-1]]
        expected_best_epoch = last['best_epoch']
        del last
    if not completed:
        run_cli('train.py', *flags)
        checkpoints = sorted(output.glob('epoch_*.pt'), key=lambda path: int(path.stem.split('_')[-1]))
        last = load_checkpoint(checkpoints[-1])
        expected_best_epoch = last['best_epoch']
        del last
    best_path = output / 'best.pt'
    best = load_checkpoint(best_path)
    purpose = 'pipeline_smoke_test' if smoke else 'model_training'
    if best['epoch'] != expected_best_epoch or best['purpose'] != purpose:
        raise RuntimeError('Snapshot best.pt belum cocok; pulihkan run dari checkpoint epoch ke folder run baru.')
    if best['dataset']['manifest_sha256'] != MANIFEST['manifest_sha256']:
        raise RuntimeError('Snapshot best.pt memakai split berbeda.')
    del best
    return {'checkpoint_sha256': file_sha256(best_path), 'manifest_sha256': MANIFEST['manifest_sha256'],
            'repo_commit': REPO_COMMIT, 'purpose': purpose}

MANIFEST = load_manifest(DATASET_DIR)
smoke_signature = {'manifest_sha256': MANIFEST['manifest_sha256'], 'repo_commit': REPO_COMMIT, 'device': DEVICE}
if SMOKE_MARKER.exists() and all(read_json(SMOKE_MARKER).get(key) == value for key, value in smoke_signature.items()):
    print('Smoke test sudah lulus untuk dataset/commit ini.')
else:
    smoke_result = run_training(WORK / 'smoke-run', 3, min(BATCH_SIZE, 8), 0, smoke=True)
    write_json(SMOKE_MARKER, {**smoke_result, **smoke_signature, 'completed_at': utc_now()})
    print('Smoke test lulus. Checkpoint smoke terpisah dari run-v2.')


## 6. Training penuh — resume otomatis dari Drive
Jalankan setelah smoke lulus. Default: 15 epoch, batch 32, ImageNet pretrained. Run parsial dilanjutkan dengan `--resume epoch_NNN.pt`; `best.pt` dipakai untuk evaluasi. Run yang sudah mencapai batas epoch/early stopping digunakan kembali. Jika GPU kehabisan memori, mulai `RUN_NAME` baru dengan batch lebih kecil; parameter checkpoint lama harus tetap sama.

In [ ]:
MANIFEST = load_manifest(DATASET_DIR)
if not SMOKE_MARKER.is_file():
    raise RuntimeError('Jalankan smoke test tahap 5 terlebih dahulu.')
smoke_result = read_json(SMOKE_MARKER)
current_smoke_signature = {'manifest_sha256': MANIFEST['manifest_sha256'], 'repo_commit': REPO_COMMIT, 'device': DEVICE}
if any(smoke_result.get(key) != value for key, value in current_smoke_signature.items()):
    raise RuntimeError('Smoke test tidak sesuai dataset/commit/device saat ini; ulangi tahap 5.')
FULL_RESULT = run_training(RUN_DIR, EPOCHS, BATCH_SIZE, WORKERS)
write_json(RUN_DIR / 'notebook-training-complete.json', {**FULL_RESULT, 'completed_at': utc_now()})
print('Training selesai; checkpoint validation terbaik:', RUN_DIR / 'best.pt')


## 7. Evaluasi — holdout test dan metrics.json
`evaluate.py` menghitung akurasi kelas/status, akurasi per kelas, confusion matrix, dan contoh salah klasifikasi. Ini evaluasi classifier sebelum gerbang HSV/confidence; mapping tiga label v2 ke status bersifat 1:1. `dataset_sources`, `composition`, `label_schema`, dan identitas split dicatat dari data. Metrik lama yang masih cocok digunakan kembali saat resume.

In [ ]:
completion = read_json(RUN_DIR / 'notebook-training-complete.json')
checkpoint_hash = file_sha256(RUN_DIR / 'best.pt')
if completion['checkpoint_sha256'] != checkpoint_hash or completion['manifest_sha256'] != MANIFEST['manifest_sha256']:
    raise RuntimeError('Checkpoint/dataset berubah; selesaikan kembali tahap 6.')
if not METRICS_PATH.exists():
    run_cli('evaluate.py', '--dataset', DATASET_DIR, '--checkpoint', RUN_DIR / 'best.pt',
            '--output', METRICS_PATH, '--batch-size', BATCH_SIZE, '--device', 'cpu')
METRICS = read_json(METRICS_PATH)
if METRICS['purpose'] != 'model_training' or METRICS['checkpoint']['sha256'] != checkpoint_hash or METRICS['manifest_sha256'] != MANIFEST['manifest_sha256']:
    raise RuntimeError('metrics.json berasal dari run lain; gunakan folder run baru.')
print(METRICS['evaluation']['context'])
print('Jumlah test:', METRICS['evaluation']['sample_count'])
print('Akurasi kelas:', METRICS['evaluation']['class_accuracy'], '| status:', METRICS['evaluation']['status_accuracy'])
print(json.dumps(METRICS['evaluation']['confusion_matrix'], indent=2, ensure_ascii=False))


## 8. Export ONNX + cek kesetaraan logits
`export_onnx.py` memeriksa model ONNX serta logits PyTorch–ONNX pada batch acak 1/2 dan satu gambar validation per label. Toleransi: `atol=1e-5`, `rtol=1e-4`. Artefak final diterbitkan script setelah parity lulus. Ekspor yang sudah cocok digunakan kembali; pasangan file tidak ditimpa.

In [ ]:
if not ONNX_PATH.exists() and not METADATA_PATH.exists():
    run_cli('export_onnx.py', '--dataset', DATASET_DIR, '--checkpoint', RUN_DIR / 'best.pt',
            '--metrics', METRICS_PATH, '--output', ONNX_PATH, '--atol', '1e-5', '--rtol', '1e-4')
if not ONNX_PATH.is_file() or not METADATA_PATH.is_file():
    raise RuntimeError('Pasangan ONNX/metadata belum lengkap; gunakan folder ekspor/run baru.')
EXPORT_METADATA = read_json(METADATA_PATH)
if (EXPORT_METADATA['purpose'] != 'model_training' or not EXPORT_METADATA['ready_for_integration']
        or EXPORT_METADATA['checkpoint_sha256'] != file_sha256(RUN_DIR / 'best.pt')
        or EXPORT_METADATA['metrics_sha256'] != file_sha256(METRICS_PATH)
        or EXPORT_METADATA['manifest_sha256'] != MANIFEST['manifest_sha256']
        or EXPORT_METADATA['onnx_sha256'] != file_sha256(ONNX_PATH)
        or not EXPORT_METADATA['export_verification']['verified']):
    raise RuntimeError('Ekspor tidak cocok dengan checkpoint/metrik/dataset saat ini; gunakan folder run baru.')
print('ONNX parity lulus:', EXPORT_METADATA['export_verification'])


## 9. Unduh empat artefak akhir
Unduhan: **model_v2.onnx**, **model_v2.metadata.json**, **metrics.json**, dan **manifest.json**. Checkpoint, history, status retraining, distribusi, serta contoh salah klasifikasi tersimpan di direktori Drive eksperimen untuk audit/reproduksi. Izinkan beberapa unduhan bila browser meminta.

Serahkan empat file ini untuk tahap integrasi model. `ready_for_integration` menyatakan pemeriksaan teknis lolos, bukan jaminan akurasi atau aktivasi model produksi.

In [ ]:
ARTIFACTS = [ONNX_PATH, METADATA_PATH, METRICS_PATH, DATASET_DIR / 'manifest.json']
if not all(path.is_file() for path in ARTIFACTS):
    raise RuntimeError('Empat artefak belum lengkap; selesaikan tahap 7–8.')
verified = read_json(METADATA_PATH)
download_manifest = load_manifest(DATASET_DIR)
if (not verified['export_verification']['verified']
        or verified['purpose'] != 'model_training'
        or verified['manifest_sha256'] != download_manifest['manifest_sha256']
        or verified['onnx_sha256'] != file_sha256(ONNX_PATH)
        or verified['metrics_sha256'] != file_sha256(METRICS_PATH)):
    raise RuntimeError('Artefak berubah sesudah export; ulangi pemeriksaan tahap 8.')
from google.colab import files
for artifact in ARTIFACTS:
    print('Mengunduh:', artifact.name)
    files.download(str(artifact))
print('Artefak juga tersimpan di:', RUN_DIR, 'dan', DATASET_DIR)
